# 3. Server Setup

Install Docker, NVIDIA Container Toolkit, and prepare the environment on the Chameleon node.

In [ ]:
from chi import server, context, lease
import os

context.version = "1.0"
context.choose_project()
context.choose_site(default="CHI@TACC")

In [ ]:
l = lease.get_lease("Serving_Proj06")
username = os.getenv('USER')
s = server.Server(f"node-serve-model-{username}")
s.refresh()
s.show(type="widget")

## Install Docker

In [ ]:
s.execute("curl -sSL https://get.docker.com/ | sudo sh")
s.execute("sudo groupadd -f docker; sudo usermod -aG docker $USER")

## Install NVIDIA Container Toolkit

In [ ]:
s.execute(
    'curl -fsSL https://nvidia.github.io/libnvidia-container/gpgkey '
    '| sudo gpg --dearmor -o /usr/share/keyrings/nvidia-container-toolkit-keyring.gpg '
    '&& curl -s -L https://nvidia.github.io/libnvidia-container/stable/deb/nvidia-container-toolkit.list '
    '| sed "s#deb https://#deb [signed-by=/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg] https://#g" '
    '| sudo tee /etc/apt/sources.list.d/nvidia-container-toolkit.list'
)
s.execute("sudo apt update")
s.execute("sudo apt-get install -y nvidia-container-toolkit")
s.execute("sudo nvidia-ctk runtime configure --runtime=docker")
s.execute(
    'sudo jq \'if has("exec-opts") then . else . + {"exec-opts": ["native.cgroupdriver=cgroupfs"]}\' '
    '/etc/docker/daemon.json | sudo tee /etc/docker/daemon.json.tmp && '
    'sudo mv /etc/docker/daemon.json.tmp /etc/docker/daemon.json'
)
s.execute("sudo systemctl restart docker")

## Install monitoring tools and set up volumes

In [ ]:
s.execute("sudo apt -y install nvtop")
s.execute("docker volume create txn_models")

## Clone project repository (private GitHub via SSH)

The repo is private, so HTTPS cloning will not work without credentials. Use SSH: generate a key on this VM, add the **public** key to GitHub (**Settings → SSH and GPG keys**), then clone.

**Step 1** — Run the next cell, then copy the printed `ssh-ed25519 ...` line.

In [ ]:
s.execute("mkdir -p ~/.ssh && chmod 700 ~/.ssh")
s.execute("test -f ~/.ssh/id_ed25519 || ssh-keygen -t ed25519 -C 'Jayraj' -f ~/.ssh/id_ed25519 -N ''")
s.execute("echo '--- Public key (add to GitHub) ---' && cat ~/.ssh/id_ed25519.pub")

**Step 2** — In GitHub: **Settings → SSH and GPG keys → New SSH key**, paste the public key, save.

**Step 3** — Run the next cell to record GitHub’s host key and clone the `Serving` branch.

In [ ]:
s.execute("ssh-keyscan -t ed25519,rsa github.com >> ~/.ssh/known_hosts 2>/dev/null || true")
s.execute("git clone --single-branch -b Serving git@github.com:RiyamPatel2001/mlops-project-proj06.git")

## Verify GPU access

In [ ]:
s.execute("nvidia-smi")

## SSH Instructions

```bash
ssh -i ~/.ssh/id_rsa_chameleon cc@<FLOATING_IP>
```